# 同态滤波与照明反射模型实验

**实验目的：**
掌握在频域中同时进行动态范围压缩和对比度增强的技术，改善光照不均。

**实验内容：**
1. 对背光阴影图像取对数，分离照射分量与反射分量。
2. 在频域设计同态滤波器，压制低频光照，提升高频细节。
3. 逆变换并取指数恢复图像，观察阴影区细节的增强。

## 一、基础知识介绍

### 1）照明-反射模型
- 图像可表示为：$I(x,y)=L(x,y)dot R(x,y)$。
- 其中 $L$ 是低频主导的照射分量，$R$ 是高频主导的反射分量。

### 2）对数变换
- 取对数后：$og I = og L + og R$，乘法关系变为加法关系。
- 这使得在频域中分别抑制低频、增强高频成为可能。

### 3）同态滤波器
- 常用形式：
$$H(u,v)=(amma_H-amma_L)eft[1-e^{-cdot D(u,v)^2/D_0^2}
ight]+amma_L$$
- 其中 $amma_L<1$ 抑制低频照明，$amma_H>1$ 增强高频细节。

## 二、应用场景

1. **背光人像增强**：压低过强背景照明，提亮暗部纹理。
2. **工业视觉检测**：减小光照不均对缺陷检测阈值的影响。
3. **医学图像预处理**：改善整体亮度不均，提高组织边缘可见性。
4. **遥感与监控图像**：在阴影区恢复更多可辨识细节。

## 三、实验部分

本实验按“准备真实图像 → 对数域分解 → 同态滤波增强 → 结果分析”的顺序进行。

- 所有代码单元前均给出说明。
- 建议按单元顺序执行。
- 若网络不可用，代码会自动回退到内置真实样本，保证可运行。

### 3.1 环境准备
导入实验依赖并设置绘图参数与中文字体回退，避免中文标题乱码。

In [ ]:
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['font.sans-serif'] = [
    'Microsoft YaHei', 'SimHei', 'SimSun',
    'Noto Sans CJK SC', 'Arial Unicode MS', 'DejaVu Sans'
]
plt.rcParams['axes.unicode_minus'] = False

### 3.2 下载并准备真实图像
本单元尝试联网下载真实照片作为实验输入；若网络失败，自动回退到内置真实样本。

In [ ]:
# 用于保存下载图像的目录
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)

# 优先尝试下载真实图像（若失败会自动回退）
image_url_candidates = [
    'https://picsum.photos/seed/homomorphic-light-2026/1024/768',
    'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg',
    'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/sudoku.png'
]

image_path = os.path.join(data_dir, 'homomorphic_real_input.png')


def download_first_available(url_candidates, file_path):
    """按顺序尝试多个 URL，任意一个成功即返回 (True, 成功URL)。"""
    for url in url_candidates:
        try:
            urllib.request.urlretrieve(url, file_path)
            return True, url
        except Exception as error:
            print(f'下载失败: {url}')
            print(f'原因: {error}')
    return False, None


def load_gray_uint8(path, max_size=768):
    """加载图像并转为 8-bit 灰度 ndarray，同时限制最大边长。"""
    image = Image.open(path).convert('L')
    width, height = image.size

    scale = min(1.0, float(max_size) / float(max(width, height)))
    if scale < 1.0:
        new_size = (int(width * scale), int(height * scale))
        resample_mode = Image.Resampling.BICUBIC if hasattr(Image, 'Resampling') else Image.BICUBIC
        image = image.resize(new_size, resample_mode)

    return np.array(image, dtype=np.uint8)


ok_image, used_url = download_first_available(image_url_candidates, image_path)

if ok_image:
    real_image = load_gray_uint8(image_path)
    image_source_note = f'online image: {used_url}'
else:
    try:
        from skimage import data
        real_image = data.camera().astype(np.uint8)
        image_source_note = 'skimage built-in fallback'
    except Exception:
        # 兜底：仅用于保证实验流程可运行
        real_image = np.tile(np.linspace(20, 230, 512, dtype=np.uint8), (512, 1))
        image_source_note = 'synthetic fallback'

# 从真实图构造“背光+阴影”输入，保证光照非均匀足够明显
rows, cols = real_image.shape
x = np.linspace(-1.0, 1.0, cols)
y = np.linspace(-1.0, 1.0, rows)
x_grid, y_grid = np.meshgrid(x, y)

illumination_field = 0.55 + 0.85 * np.exp(-((x_grid - 0.45) ** 2 + (y_grid + 0.05) ** 2) / 0.35)
illumination_field += 0.18 * x_grid
illumination_field = np.clip(illumination_field, 0.30, 1.35)

backlit_shadow_image = np.clip(real_image.astype(np.float32) * illumination_field, 0, 255).astype(np.uint8)

print('图像来源:', image_source_note)
print('真实图尺寸:', real_image.shape)
print('待处理背光阴影图尺寸:', backlit_shadow_image.shape)

### 3.3 展示真实图与背光阴影输入
先观察光照不均对灰度分布的影响，并通过直方图建立直观认识。

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].imshow(real_image, vmin=0, vmax=255)
axes[0, 0].set_title('真实原图')
axes[0, 0].axis('off')

axes[0, 1].imshow(backlit_shadow_image, vmin=0, vmax=255)
axes[0, 1].set_title('背光阴影输入图')
axes[0, 1].axis('off')

axes[1, 0].hist(real_image.ravel(), bins=256, range=(0, 255), color='steelblue', alpha=0.85)
axes[1, 0].set_title('真实原图直方图')
axes[1, 0].set_xlabel('灰度级')
axes[1, 0].set_ylabel('像素数量')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].hist(backlit_shadow_image.ravel(), bins=256, range=(0, 255), color='darkorange', alpha=0.85)
axes[1, 1].set_title('背光阴影输入图直方图')
axes[1, 1].set_xlabel('灰度级')
axes[1, 1].set_ylabel('像素数量')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.4 同态滤波核心函数
本单元实现：
1. 对数域转换与归一化；
2. 照射/反射分量的频域分离；
3. 同态高通滤波器构造与增强结果恢复。

In [ ]:
def to_float_gray(image_uint8):
    """uint8 灰度图转 0~1 浮点图。"""
    return image_uint8.astype(np.float32) / 255.0


def safe_log(image_float, eps=1e-6):
    """安全对数，避免 log(0)。"""
    return np.log(image_float + eps)


def normalize_to_uint8(image_float):
    """将浮点图线性归一化为 uint8。"""
    image_float = np.real(image_float)
    min_value = np.min(image_float)
    max_value = np.max(image_float)

    if max_value - min_value < 1e-12:
        return np.zeros_like(image_float, dtype=np.uint8)

    norm = (image_float - min_value) / (max_value - min_value)
    return np.clip(norm * 255.0, 0, 255).astype(np.uint8)


def gaussian_lowpass(shape, sigma=35.0):
    """构造频域高斯低通滤波器。"""
    rows, cols = shape
    u = np.arange(rows) - rows / 2.0
    v = np.arange(cols) - cols / 2.0
    v_grid, u_grid = np.meshgrid(v, u)
    distance2 = u_grid ** 2 + v_grid ** 2
    return np.exp(-distance2 / (2.0 * sigma * sigma))


def homomorphic_filter(shape, gamma_low=0.6, gamma_high=1.9, c=1.0, d0=45.0):
    """构造同态滤波器 H(u,v)。"""
    rows, cols = shape
    u = np.arange(rows) - rows / 2.0
    v = np.arange(cols) - cols / 2.0
    v_grid, u_grid = np.meshgrid(v, u)
    distance2 = u_grid ** 2 + v_grid ** 2

    h = (gamma_high - gamma_low) * (1.0 - np.exp(-c * distance2 / (d0 * d0))) + gamma_low
    return h


def decompose_illumination_reflectance(image_uint8, sigma=35.0):
    """在对数域估计照射与反射分量。"""
    image_float = to_float_gray(image_uint8)
    log_image = safe_log(image_float)

    spectrum = np.fft.fftshift(np.fft.fft2(log_image))
    lowpass = gaussian_lowpass(log_image.shape, sigma=sigma)

    log_illumination = np.real(np.fft.ifft2(np.fft.ifftshift(spectrum * lowpass)))
    log_reflectance = log_image - log_illumination

    illumination = np.exp(log_illumination)
    reflectance = np.exp(log_reflectance)

    return {
        'log_image': log_image,
        'log_illumination': log_illumination,
        'log_reflectance': log_reflectance,
        'illumination': illumination,
        'reflectance': reflectance,
        'lowpass': lowpass
    }


def apply_homomorphic_enhancement(image_uint8, gamma_low=0.6, gamma_high=1.9, c=1.0, d0=45.0):
    """执行同态滤波：log -> 频域滤波 -> 逆变换 -> exp。"""
    image_float = to_float_gray(image_uint8)
    log_image = safe_log(image_float)

    spectrum = np.fft.fftshift(np.fft.fft2(log_image))
    filter_h = homomorphic_filter(log_image.shape, gamma_low=gamma_low, gamma_high=gamma_high, c=c, d0=d0)

    filtered_spectrum = spectrum * filter_h
    filtered_log = np.real(np.fft.ifft2(np.fft.ifftshift(filtered_spectrum)))
    restored = np.exp(filtered_log)

    enhanced_uint8 = normalize_to_uint8(restored)

    return enhanced_uint8, {
        'log_image': log_image,
        'filter_h': filter_h,
        'filtered_log': filtered_log,
        'restored': restored
    }


def cdf_from_hist(histogram):
    """由直方图计算 CDF。"""
    pdf = histogram.astype(np.float64) / np.sum(histogram)
    return np.cumsum(pdf)

### 3.5 实验1执行：对数域分离照射与反射
观察低频照射项与高频反射项的视觉差异，验证模型分解是否合理。

In [ ]:
decomposition = decompose_illumination_reflectance(backlit_shadow_image, sigma=38.0)

log_vis = normalize_to_uint8(decomposition['log_image'])
illumination_vis = normalize_to_uint8(decomposition['illumination'])
reflectance_vis = normalize_to_uint8(decomposition['reflectance'])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].imshow(backlit_shadow_image, vmin=0, vmax=255)
axes[0].set_title('背光阴影输入图')
axes[0].axis('off')

axes[1].imshow(log_vis, vmin=0, vmax=255)
axes[1].set_title('对数域图像')
axes[1].axis('off')

axes[2].imshow(illumination_vis, vmin=0, vmax=255)
axes[2].set_title('估计照射分量（低频）')
axes[2].axis('off')

axes[3].imshow(reflectance_vis, vmin=0, vmax=255)
axes[3].set_title('估计反射分量（高频）')
axes[3].axis('off')

plt.tight_layout()
plt.show()

### 3.6 实验2执行：同态滤波增强与分布对比
输出增强前后图像、差分图、直方图、CDF 及滤波器响应，并量化阴影区细节变化。

In [ ]:
enhanced_image, homomorphic_info = apply_homomorphic_enhancement(
    backlit_shadow_image,
    gamma_low=0.55,
    gamma_high=2.00,
    c=1.10,
    d0=42.0
)

difference_map = enhanced_image.astype(np.int16) - backlit_shadow_image.astype(np.int16)

hist_before, _ = np.histogram(backlit_shadow_image.ravel(), bins=256, range=(0, 256))
hist_after, _ = np.histogram(enhanced_image.ravel(), bins=256, range=(0, 256))
cdf_before = cdf_from_hist(hist_before)
cdf_after = cdf_from_hist(hist_after)
x = np.arange(256)

shadow_threshold = np.percentile(backlit_shadow_image, 30)
shadow_mask = backlit_shadow_image <= shadow_threshold

shadow_std_before = float(np.std(backlit_shadow_image[shadow_mask]))
shadow_std_after = float(np.std(enhanced_image[shadow_mask]))
shadow_mean_before = float(np.mean(backlit_shadow_image[shadow_mask]))
shadow_mean_after = float(np.mean(enhanced_image[shadow_mask]))

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

axes[0, 0].imshow(backlit_shadow_image, vmin=0, vmax=255)
axes[0, 0].set_title('增强前（背光阴影图）')
axes[0, 0].axis('off')

axes[0, 1].imshow(enhanced_image, vmin=0, vmax=255)
axes[0, 1].set_title('同态滤波后')
axes[0, 1].axis('off')

im = axes[0, 2].imshow(difference_map, cmap='bwr')
axes[0, 2].set_title('差分图（后-前）')
axes[0, 2].axis('off')
plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

axes[1, 0].plot(x, hist_before, label='增强前', linewidth=1.6)
axes[1, 0].plot(x, hist_after, label='增强后', linewidth=1.6)
axes[1, 0].set_title('直方图对比')
axes[1, 0].set_xlabel('灰度级')
axes[1, 0].set_ylabel('像素数量')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(x, cdf_before, label='增强前 CDF', linewidth=2.0)
axes[1, 1].plot(x, cdf_after, label='增强后 CDF', linewidth=2.0)
axes[1, 1].set_title('CDF 对比')
axes[1, 1].set_xlabel('灰度级')
axes[1, 1].set_ylabel('累计概率')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

filter_show = axes[1, 2].imshow(homomorphic_info['filter_h'], cmap='viridis')
axes[1, 2].set_title('同态滤波器 H(u,v)')
axes[1, 2].axis('off')
plt.colorbar(filter_show, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print('阴影区统计（取输入图最暗 30% 像素）:')
print(f'阴影均值: 增强前={shadow_mean_before:.2f}, 增强后={shadow_mean_after:.2f}')
print(f'阴影标准差: 增强前={shadow_std_before:.2f}, 增强后={shadow_std_after:.2f}')
print('说明：阴影区标准差增大通常意味着局部细节与对比度得到增强。')

### 3.7 参数影响：不同截止频率对结果的影响
通过调整 $D_0$ 观察“增强强度 vs 伪影风险”的平衡。

In [ ]:
parameter_groups = [
    ('D0=25, gammaH=1.8', dict(gamma_low=0.60, gamma_high=1.80, c=1.00, d0=25.0)),
    ('D0=42, gammaH=2.0', dict(gamma_low=0.55, gamma_high=2.00, c=1.10, d0=42.0)),
    ('D0=80, gammaH=2.2', dict(gamma_low=0.50, gamma_high=2.20, c=1.00, d0=80.0))
]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for idx, (name, params) in enumerate(parameter_groups):
    result_img, _ = apply_homomorphic_enhancement(backlit_shadow_image, **params)
    axes[idx].imshow(result_img, vmin=0, vmax=255)
    axes[idx].set_title(name)
    axes[idx].axis('off')

plt.suptitle('同态滤波参数敏感性对比', y=1.03)
plt.tight_layout()
plt.show()

## 四、总结与思考

### 实验结论
1. 对数变换后，照射与反射分量在频域中具备可分离性。
2. 同态滤波通过抑制低频照明并增强高频细节，可改善光照不均问题。
3. 从增强前后直方图与 CDF 可见：灰度分布更均衡，暗部可辨识细节增加。
4. 参数过强会带来噪声放大与边缘过冲，需结合任务场景调参。

### 思考题
1. 为什么 $amma_H$ 过大时会让图像看起来“发硬”或噪声明显？
2. 同态滤波与 CLAHE 在“阴影增强”方面各自的优势是什么？
3. 若处理彩色图像，应在 RGB 三通道分别处理，还是仅处理亮度通道？
4. 在实时视频场景中，如何降低 FFT 带来的计算开销？